# PyTorch Autograd 

**Reference Lecture:** [PyTorch Autograd | Video 3 | CampusX](http://www.youtube.com/watch?v=BECZ0UB5AR0)

## 1. What is Autograd & Why Do We Need It?
**Autograd** is PyTorch's automatic differentiation engine that powers neural network training.

### The Core Problem: Nested Functions & Chain Rule
Neural networks are deeply nested composite functions:
* **Forward Pass:** Inputs ($x$) and parameters ($w, b$) pass through linear combinations ($z = wx + b$), activations ($\hat{y} = \sigma(z)$), and a loss function ($L$).
* **Backward Pass (Backpropagation):** Training via Gradient Descent requires computing partial derivatives of the loss with respect to every weight and bias ($\frac{\partial L}{\partial w}, \frac{\partial L}{\partial b}$) using the **Chain Rule**.
* Calculating and hardcoding analytical derivatives manually becomes virtually impossible as networks scale to millions of parameters. Autograd automates this entire process.

### How Autograd Works Under the Hood (DAG)
1. Setting `requires_grad=True` on a tensor instructs PyTorch to track every operation performed on it.
2. During the forward pass, PyTorch dynamically builds a **Directed Acyclic Graph (DAG)**:
   * **Leaves:** Input/parameter tensors with `requires_grad=True` (e.g., weights $w$, biases $b$).
   * **Intermediate Nodes:** Operations and intermediate tensors (e.g., $z$, $\hat{y}$), each storing a `grad_fn` pointer to its backward derivative formula.
   * **Root:** The final output scalar tensor (typically `loss`).
3. Calling `.backward()` on the root traverses the DAG in reverse from root to leaves, applying the chain rule and populating `.grad` on all leaf tensors.

## 2. Example 1: Simple Function ($y = x^2$)
Compute $\frac{dy}{dx} = 2x$ at $x = 3$ manually first, then verify using PyTorch Autograd.

In [1]:
# Manual analytical derivative
def dy_dx(x):
    return 2 * x

print("Manual dy/dx at x=3:", dy_dx(3))

# Using PyTorch Autograd
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2

print("x:", x)
print("y:", y)  # Notice grad_fn=<PowBackward0>

# Compute gradients via backward pass
y.backward()
print("Autograd x.grad:", x.grad)

Manual dy/dx at x=3: 6
x: tensor(3., requires_grad=True)
y: tensor(9., grad_fn=<PowBackward0>)
Autograd x.grad: tensor(6.)


## 3. Example 2: Nested Function & Chain Rule ($z = \sin(x^2)$)
By the chain rule:
$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx} = \cos(y) \cdot 2x = 2x \cos(x^2)$$

* **Leaf Tensor vs Intermediate Tensor:** PyTorch only populates `.grad` for **leaf tensors** (`x`). Accessing `.grad` on a non-leaf intermediate tensor (`y`) triggers a warning and returns `None` (unless `.retain_grad()` is explicitly called).

In [2]:
import math

# Manual chain rule derivative
def dz_dx(x):
    return 2 * x * math.cos(x**2)

print("Manual dz/dx at x=4:", dz_dx(4))

# Using PyTorch Autograd
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

print("x:", x)
print("y:", y)
print("z:", z)

# Trigger backpropagation from the root tensor z
z.backward()
print("Autograd x.grad:", x.grad)

# Non-leaf tensor warning check (y is an intermediate node)
print("Intermediate y.grad:", y.grad)

Manual dz/dx at x=4: -7.661275842587077
x: tensor(4., requires_grad=True)
y: tensor(16., grad_fn=<PowBackward0>)
z: tensor(-0.2879, grad_fn=<SinBackward0>)
Autograd x.grad: tensor(-7.6613)
Intermediate y.grad: None


/tmp/ipykernel_1150/3843385777.py:23: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /__w/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:488.)
  print("Intermediate y.grad:", y.grad)


## 4. Example 3: Neural Network Training Step (Binary Classification)
Consider a single-neuron perceptron predicting placement (`y = 0`) from CGPA (`x = 6.7`), initialized with `w = 1.0` and `b = 0.0`:

1. **Linear step:** $z = w \cdot x + b$
2. **Activation:** $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$
3. **Loss (Binary Cross-Entropy):** $L = -\left[y \ln(\hat{y}) + (1 - y)\ln(1 - \hat{y})\right]$

### Part A: Manual Backpropagation
Computing gradients manually requires differentiating each step and multiplying via the chain rule:
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}, \qquad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial b}$$

In [3]:
# Inputs and initial parameters (without Autograd)
x = torch.tensor(6.7)  # Input feature (CGPA)
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

# Forward pass
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)
print("Loss:", loss)

# Manual Backward pass (Chain Rule)
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))
dy_pred_dz = y_pred * (1 - y_pred)
dz_dw = x
dz_db = 1

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db):   {dL_db}")

Loss: tensor(6.7012)
Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db):   0.998770534992218


### Part B: Automatic Backpropagation with Autograd
By setting `requires_grad=True` on the learnable parameters (`w` and `b`)   while keeping `requires_grad=False` on fixed inputs (`x` and `y`) a single `loss.backward()` call computes both gradients automatically.

In [4]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

# Enable gradient tracking only on learnable parameters
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Forward pass
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

print("z:", z)
print("y_pred:", y_pred)
print("loss:", loss)

# Backward pass
loss.backward()

print("Autograd w.grad:", w.grad)
print("Autograd b.grad:", b.grad)

z: tensor(6.7000, grad_fn=<AddBackward0>)
y_pred: tensor(0.9988, grad_fn=<SigmoidBackward0>)
loss: tensor(6.7012, grad_fn=<NegBackward0>)
Autograd w.grad: tensor(6.6918)
Autograd b.grad: tensor(0.9988)


## 5. Autograd with Vector Inputs
Autograd works seamlessly on multi-dimensional tensors. For a vector $\mathbf{x} = [x_1, x_2, x_3]$ and scalar output $y = \frac{1}{3}(x_1^2 + x_2^2 + x_3^2)$, calling `y.backward()` populates `x.grad` with the partial derivative vector:
$$\nabla_{\mathbf{x}} y = \left[\frac{2x_1}{3}, \; \frac{2x_2}{3}, \; \frac{2x_3}{3}\right]$$

In [5]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x**2).mean()

print("x:", x)
print("y:", y)

y.backward()
print("x.grad:", x.grad)  # Expected: [2/3, 4/3, 6/3] -> tensor([0.6667, 1.3333, 2.0000])

x: tensor([1., 2., 3.], requires_grad=True)
y: tensor(4.6667, grad_fn=<MeanBackward0>)
x.grad: tensor([0.6667, 1.3333, 2.0000])


## 6. Clearing Gradients (`grad.zero_()`)
By default, PyTorch **accumulates (sums) gradients** into `.grad` every time `.backward()` is called. 
* In a multi-epoch training loop, failing to reset gradients causes new gradients to add onto previous epochs' gradients, corrupting parameter updates.
* Always zero out gradients in-place using `x.grad.zero_()` (or `optimizer.zero_grad()`) before the next backward pass.

In [6]:
x = torch.tensor(2.0, requires_grad=True)

# Pass 1
y = x ** 2
y.backward()
print("After Pass 1, x.grad:", x.grad)  # 4.0

# Pass 2 WITHOUT clearing (accumulates: 4.0 + 4.0 = 8.0)
y = x ** 2
y.backward()
print("After Pass 2 (without zero_), x.grad:", x.grad)

# Clear gradients in-place
x.grad.zero_()
print("After x.grad.zero_(), x.grad:", x.grad)

# Pass 3 AFTER clearing (clean gradient: 4.0)
y = x ** 2
y.backward()
print("After Pass 3 (with zero_), x.grad:", x.grad)

After Pass 1, x.grad: tensor(4.)
After Pass 2 (without zero_), x.grad: tensor(8.)
After x.grad.zero_(), x.grad: tensor(0.)
After Pass 3 (with zero_), x.grad: tensor(4.)


## 7. Disabling Gradient Tracking
Once a model is trained and used for **inference/prediction**, backpropagation is unnecessary. Keeping gradient tracking active wastes memory and slows down computation. PyTorch provides three ways to turn off gradient tracking:

1. **`x.requires_grad_(False)`:** Disables gradient tracking on the tensor in-place.
2. **`z = x.detach()`:** Creates a new tensor sharing the same data but detached from the computation graph.
3. **`with torch.no_grad():` (Recommended):** Context manager that temporarily disables gradient tracking for all operations inside the block without mutating tensor flags.

In [7]:
# Option 1: In-place modification via requires_grad_(False)
x = torch.tensor(2.0, requires_grad=True)
x.requires_grad_(False)
y = x ** 2
print("Option 1 - y without grad_fn:", y)
# Calling y.backward() here would raise RuntimeError: element 0 of tensors does not require grad


Option 1 - y without grad_fn: tensor(4.)


In [8]:
# Option 2: Detaching into a new tensor via detach()
x = torch.tensor(2.0, requires_grad=True)
z = x.detach()

y = x ** 2    # Tracked in computation graph
y1 = z ** 2   # Detached from computation graph

y.backward()
print("Option 2 - y has grad_fn:", y)
print("Option 2 - y1 has no grad_fn:", y1)
# Calling y1.backward() here would raise RuntimeError

Option 2 - y has grad_fn: tensor(4., grad_fn=<PowBackward0>)
Option 2 - y1 has no grad_fn: tensor(4.)


In [9]:
# Option 3: Context manager (most common in evaluation/inference loops)
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2
    print("Inside torch.no_grad() -> y:", y, "| requires_grad:", y.requires_grad)

# Outside the block, x still retains requires_grad=True for future training steps
y_train = x ** 2
y_train.backward()
print("Outside torch.no_grad() -> x.grad:", x.grad)

Inside torch.no_grad() -> y: tensor(4.) | requires_grad: False
Outside torch.no_grad() -> x.grad: tensor(4.)
